# Live Demo — Clean & Build Panel
### Session 5 · ExInt II · WU Vienna · SS 2026

**What we do in this notebook:**
1. Find the most recent pull folder automatically
2. Read and concatenate all parquet chunks
3. Inspect the raw combined data
4. Drop duplicates and rows missing panel identifiers
5. Convert numeric columns
6. Sort into a firm-year panel
7. Save as `data/processed/panel_clean.parquet`

This is the **mini version** of `02_clean.py` — same logic, using the 5-firm demo data from the pull notebook.

> **Reference project:** https://github.com/vkiefner/sme-intl

---
## Cell 1 — Imports

In [3]:
import os
from pathlib import Path
from dotenv import load_dotenv
from pathlib import Path
from datetime import datetime
import pandas as pd

def find_env():
    current = Path(os.getcwd())
    for path in [current] + list(current.parents):
        if (path / ".env").exists():
            return path / ".env"
        for sibling in path.iterdir():
            if sibling.is_dir() and (sibling / ".env").exists():
                return sibling / ".env"
    raise FileNotFoundError("Could not find .env anywhere.")

env_file = find_env()
project_root = env_file.parent
os.chdir(project_root)

print(f"Working directory: {os.getcwd()}")
print(f"pandas: {pd.__version__}")

Working directory: c:\Users\vkiefner\dev\python\venv1\ResearchDesign in SME Research\sme-internationalization
pandas: 2.2.1


---
## Cell 2 — Find the most recent pull folder

Folders are named `YYYY-MM-DD_HH-MM-SS` so alphabetical sort = chronological sort.  
We always pick the last one — no manual path editing needed.

In [4]:
RAW_DIR  = Path("data") / "raw"
PROC_DIR = Path("data") / "processed"
PROC_DIR.mkdir(parents=True, exist_ok=True)

# List all timestamped pull folders
folders = sorted([f for f in RAW_DIR.iterdir() if f.is_dir()])

print("Pull folders found:")
for f in folders:
    files = list(f.glob("fyear_*.parquet"))
    print(f"  {f.name}  ({len(files)} parquet files)")

# Pick the most recent
latest = folders[-1]
print(f"\nUsing: {latest.name}")

Pull folders found:
  2026-05-26_12-17-58  (10 parquet files)
  2026-05-26_12-48-27  (10 parquet files)

Using: 2026-05-26_12-48-27


---
## Cell 3 — Read all parquet chunks

In [5]:
parquet_files = sorted(latest.glob("fyear_*.parquet"))
print(f"Reading {len(parquet_files)} files...\n")

chunks = []
for f in parquet_files:
    chunk = pd.read_parquet(f)
    chunks.append(chunk)
    print(f"  {f.name:<25}  {len(chunk):>4} rows")

# Concatenate into one DataFrame
df = pd.concat(chunks, ignore_index=True)
print(f"\nCombined: {df.shape[0]} rows × {df.shape[1]} columns")

Reading 10 files...

  fyear_2015.parquet            5 rows
  fyear_2016.parquet            5 rows
  fyear_2017.parquet            5 rows
  fyear_2018.parquet            5 rows
  fyear_2019.parquet            5 rows
  fyear_2020.parquet            5 rows
  fyear_2021.parquet            5 rows
  fyear_2022.parquet            5 rows
  fyear_2023.parquet            5 rows
  fyear_2024.parquet            5 rows

Combined: 50 rows × 444 columns


---
## Cell 4 — First look at the combined data

In [6]:
df.head(10)

,gvkey,indfmt,datafmt,consol,popsrc,acctstd,acqmeth,bspr,compst,curcd,...,conm,costat,fic,loc,naicsh,sich,rank,au,auop,hiid
0,014447,INDL,HIST_STD,C,I,DI,None,GO,None,EUR,...,LVMH MOET HENNESSY LOUIS V,A,FRA,FRA,3152,2330,1,5,4,02W
1,016603,INDL,HIST_STD,C,I,DI,None,GO,None,CHF,...,NESTLE SA/AG,A,CHE,CHE,311,2000,1,6,1,01W
2,017436,INDL,HIST_STD,C,I,DI,None,GO,None,EUR,...,BASF SE,A,DEU,DEU,325,2800,1,6,1,01W
3,024625,INDL,HIST_STD,C,I,DI,None,GO,None,USD,...,TOTALENERGIES SE,A,FRA,FRA,324110,2911,1,4,1,02W
4,100737,INDL,HIST_STD,C,I,DI,None,GO,None,EUR,...,VOLKSWAGEN AG,A,DEU,DEU,336111,3711,1,7,4,02W
5,014447,INDL,HIST_STD,C,I,DI,None,GO,None,EUR,...,LVMH MOET HENNESSY LOUIS V,A,FRA,FRA,3152,2330,1,9,4,02W
6,016603,INDL,HIST_STD,C,I,DI,None,GO,None,CHF,...,NESTLE SA/AG,A,CHE,CHE,311,2000,1,6,1,01W
7,017436,INDL,HIST_STD,C,I,DI,None,GO,None,EUR,...,BASF SE,A,DEU,DEU,325,2800,1,6,1,01W
8,024625,INDL,HIST_STD,C,I,DI,None,GO,None,USD,...,TOTALENERGIES SE,A,FRA,FRA,324110,2911,1,4,1,02W
9,100737,INDL,HIST_STD,C,I,DI,None,GO,None,EUR,...,VOLKSWAGEN AG,A,DEU,DEU,336111,3711,1,7,4,02W


In [7]:
# Data types overview
print(f"Shape: {df.shape}")
print(f"\nDtype counts:")
print(df.dtypes.value_counts())

Shape: (50, 444)

Dtype counts:
object            279
float64           153
int64              11
datetime64[ns]      1
Name: count, dtype: int64


In [8]:
# Panel coverage — firms × years
df.groupby(["gvkey", "conm"])["fyear"].agg(["min", "max", "count"]).rename(
    columns={"min": "first_year", "max": "last_year", "count": "n_years"}
)

,,first_year,last_year,n_years
gvkey,conm,,,
014447,LVMH MOET HENNESSY LOUIS V,2015,2024,10
016603,NESTLE SA/AG,2015,2024,10
017436,BASF SE,2015,2024,10
024625,TOTALENERGIES SE,2015,2024,10
100737,VOLKSWAGEN AG,2015,2024,10


---
## Cell 5 — Check for duplicate rows

In [9]:
n_dupes = df.duplicated().sum()
print(f"Exact duplicate rows: {n_dupes}")

# Also check: are there duplicate gvkey-fyear combinations?
n_key_dupes = df.duplicated(subset=["gvkey", "fyear"]).sum()
print(f"Duplicate gvkey-fyear pairs: {n_key_dupes}")

# Drop exact duplicates
df = df.drop_duplicates()
print(f"\nAfter dedup: {len(df)} rows")

Exact duplicate rows: 0
Duplicate gvkey-fyear pairs: 0

After dedup: 50 rows


---
## Cell 6 — Drop rows missing panel identifiers

Every row in a firm-year panel **must** have a firm identifier (`gvkey`) and a time identifier (`fyear`).  
Without both, we cannot place the observation in the panel.

In [10]:
n_before = len(df)

df = df.dropna(subset=["gvkey", "fyear"])

n_dropped = n_before - len(df)
print(f"Dropped {n_dropped} rows missing gvkey or fyear")
print(f"Remaining: {len(df)} rows")

Dropped 0 rows missing gvkey or fyear
Remaining: 50 rows


---
## Cell 7 — Convert object columns to numeric where possible

WRDS sometimes returns numeric fields as `object` (string) dtype.  
We use `pd.to_numeric(..., errors='coerce')` — this converts what it can and turns non-numeric values into `NaN` (never crashes).

In [11]:
# Columns we know are strings — leave these alone
STRING_COLS = {
    "gvkey", "conm", "cusip", "isin", "sedol", "tic",
    "naics", "sic", "loc", "curcd", "fic", "exchg",
    "costat", "stalt", "datafmt", "indfmt", "popsrc", "consol"
}

converted = []
for col in df.columns:
    if df[col].dtype == object and col not in STRING_COLS:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        converted.append(col)

print(f"Converted {len(converted)} columns to numeric dtype")
if converted:
    print("Columns converted:", converted)

Converted 266 columns to numeric dtype
Columns converted: ['acctstd', 'acqmeth', 'bspr', 'compst', 'final', 'fdate', 'pdate', 'accli', 'acco', 'acofs', 'acoxfs', 'acqdisn', 'acqdiso', 'adpac', 'amdc', 'apalch', 'apdpfs', 'apfs', 'apofs', 'artfs', 'asdis', 'asinv', 'bcef', 'bct', 'ca', 'capcst', 'capfl', 'capr1', 'capr2', 'capr3', 'caprt', 'capxfi', 'cfbd', 'cfere', 'cfo', 'cfpdo', 'cga', 'chefs', 'chenfd', 'chfs', 'chs', 'cmp', 'crvnli', 'dbtb', 'dbte', 'dcsfd', 'dcufd', 'dd1fs', 'dfpac', 'dispoch', 'dlcfs', 'dltis', 'dltr', 'doc', 'dpdc', 'dpltb', 'dpsc', 'dpstb', 'dptb', 'dptc', 'dptic', 'dvp', 'dvpdp', 'dvrec', 'dvrre', 'dvsco', 'eiea', 'eqdivp', 'exres', 'exreu', 'fatb', 'fate', 'fatl', 'fatp', 'fdfr', 'fea', 'fel', 'ffs', 'fininc', 'finle', 'finre', 'finvao', 'fsrco', 'fsrcopo', 'fsrcopt', 'fsrct', 'fuseo', 'fuset', 'iaeq', 'iafxi', 'ialoi', 'ialti', 'iamli', 'iaoi', 'iapli', 'iarei', 'iassi', 'iasti', 'iati', 'ibki', 'ibmii', 'idiis', 'idilb', 'idilc', 'idis', 'idist', 'idits', '

---
## Cell 8 — Ensure `fyear` is integer

In [12]:
df["fyear"] = df["fyear"].astype(int)
print(f"fyear dtype: {df['fyear'].dtype}")
print(f"fyear range: {df['fyear'].min()} – {df['fyear'].max()}")

fyear dtype: int32
fyear range: 2015 – 2024


---
## Cell 9 — Sort into a firm-year panel

In [13]:
df = df.sort_values(["gvkey", "fyear"]).reset_index(drop=True)

print("Sorted by gvkey, fyear. First 10 rows:")
df[["gvkey", "conm", "fyear", "loc", "at", "sale", "ib", "xrd", "emp"]].head(10)

Sorted by gvkey, fyear. First 10 rows:


,gvkey,conm,fyear,loc,at,sale,ib,xrd,emp
0,014447,LVMH MOET HENNESSY LOUIS V,2015,FRA,57601.0,35664.0,3573.0,97.0,100.983
1,014447,LVMH MOET HENNESSY LOUIS V,2016,FRA,59622.0,37600.0,3981.0,111.0,107.053
2,014447,LVMH MOET HENNESSY LOUIS V,2017,FRA,68550.0,42636.0,5129.0,130.0,145.247
3,014447,LVMH MOET HENNESSY LOUIS V,2018,FRA,74300.0,46826.0,6354.0,130.0,127.739
4,014447,LVMH MOET HENNESSY LOUIS V,2019,FRA,96507.0,53670.0,7171.0,140.0,163.309
5,014447,LVMH MOET HENNESSY LOUIS V,2020,FRA,108671.0,44651.0,4702.0,139.0,148.343
6,014447,LVMH MOET HENNESSY LOUIS V,2021,FRA,125311.0,64215.0,12036.0,147.0,157.953
7,014447,LVMH MOET HENNESSY LOUIS V,2022,FRA,134646.0,79184.0,14084.0,172.0,173.492
8,014447,LVMH MOET HENNESSY LOUIS V,2023,FRA,143694.0,86153.0,15174.0,NaN,192.287
9,014447,LVMH MOET HENNESSY LOUIS V,2024,FRA,149190.0,84683.0,12550.0,NaN,200.518


---
## Cell 10 — Missing value summary

In [14]:
# How complete is each column?
missing = pd.DataFrame({
    "missing_n":   df.isnull().sum(),
    "missing_pct": (df.isnull().sum() / len(df) * 100).round(1),
    "dtype":       df.dtypes
})

# Show columns with at least one missing value, sorted by % missing
missing[missing["missing_n"] > 0].sort_values("missing_pct", ascending=False)

,missing_n,missing_pct,dtype
acctstd,50,100.0,float64
oancfd,50,100.0,float64
pliach,50,100.0,float64
pcl,50,100.0,float64
pacqp,50,100.0,float64
...,...,...,...
eieac,8,16.0,float64
xrent,7,14.0,float64
am,4,8.0,float64
xrd,2,4.0,float64


In [15]:
# How complete are the key variables students will use?
key_vars = ["at", "sale", "ib", "xrd", "emp", "dltt", "pifo"]
available_key = [v for v in key_vars if v in df.columns]

print("Completeness of key variables:")
for col in available_key:
    pct_complete = (df[col].notna().sum() / len(df) * 100)
    bar = "█" * int(pct_complete / 5)
    print(f"  {col:<6}  {pct_complete:>5.1f}%  {bar}")

Completeness of key variables:
  at      100.0%  ████████████████████
  sale    100.0%  ████████████████████
  ib      100.0%  ████████████████████
  xrd      96.0%  ███████████████████
  emp     100.0%  ████████████████████
  dltt    100.0%  ████████████████████


---
## Cell 11 — Panel statistics

In [16]:
print("="*45)
print("Panel Statistics")
print("="*45)
print(f"  Total firm-years:   {len(df):>8,}")
print(f"  Unique firms:       {df['gvkey'].nunique():>8,}")
print(f"  Years covered:      {df['fyear'].min()}–{df['fyear'].max()}")
print(f"  Countries:          {df['loc'].nunique():>8,}")
print(f"  Total columns:      {df.shape[1]:>8,}")

print("\n  Observations per year:")
for year, count in df.groupby("fyear").size().items():
    print(f"    {year}: {count:>4}")

Panel Statistics
  Total firm-years:         50
  Unique firms:              5
  Years covered:      2015–2024
  Countries:                 3
  Total columns:           444

  Observations per year:
    2015:    5
    2016:    5
    2017:    5
    2018:    5
    2019:    5
    2020:    5
    2021:    5
    2022:    5
    2023:    5
    2024:    5


---
## Cell 12 — Quick descriptive stats on key variables

In [17]:
df[available_key].describe().round(2)

,at,sale,ib,xrd,emp,dltt
count,50.00,50.00,50.00,48.00,50.00,50.00
mean,215752.36,129264.46,8732.56,3766.79,262.42,42184.92
std,162111.23,81028.98,5964.94,5083.57,212.11,36496.18
min,57601.00,35664.00,-7242.00,97.00,95.39,3932.00
25%,87058.25,66170.50,4798.25,877.25,107.88,15975.00
50%,136098.50,90572.50,8966.50,1682.50,153.15,27439.50
75%,285028.75,192866.00,12470.50,2172.50,319.25,47530.25
max,632905.00,324656.00,21384.00,17963.00,684.02,137061.00


---
## Cell 13 — Save clean panel as parquet

In [18]:
out_path = PROC_DIR / "panel_clean.parquet"

df.to_parquet(out_path, index=False)

size_mb = out_path.stat().st_size / 1_048_576
print(f"Saved: {out_path}")
print(f"Size:  {size_mb:.2f} MB")
print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")

Saved: data\processed\panel_clean.parquet
Size:  0.26 MB
Shape: 50 rows × 444 columns


---
## Cell 14 — Verify: read back and confirm

In [19]:
check = pd.read_parquet(out_path)

print(f"Read back: {check.shape[0]} rows × {check.shape[1]} columns")
print(f"dtypes preserved: {check.dtypes.value_counts().to_dict()}")
check[["gvkey", "conm", "fyear", "at", "sale", "ib"]].head(10)

Read back: 50 rows × 444 columns
dtypes preserved: {dtype('float64'): 417, dtype('O'): 13, dtype('int64'): 12, dtype('int32'): 1, dtype('<M8[ns]'): 1}


,gvkey,conm,fyear,at,sale,ib
0,014447,LVMH MOET HENNESSY LOUIS V,2015,57601.0,35664.0,3573.0
1,014447,LVMH MOET HENNESSY LOUIS V,2016,59622.0,37600.0,3981.0
2,014447,LVMH MOET HENNESSY LOUIS V,2017,68550.0,42636.0,5129.0
3,014447,LVMH MOET HENNESSY LOUIS V,2018,74300.0,46826.0,6354.0
4,014447,LVMH MOET HENNESSY LOUIS V,2019,96507.0,53670.0,7171.0
5,014447,LVMH MOET HENNESSY LOUIS V,2020,108671.0,44651.0,4702.0
6,014447,LVMH MOET HENNESSY LOUIS V,2021,125311.0,64215.0,12036.0
7,014447,LVMH MOET HENNESSY LOUIS V,2022,134646.0,79184.0,14084.0
8,014447,LVMH MOET HENNESSY LOUIS V,2023,143694.0,86153.0,15174.0
9,014447,LVMH MOET HENNESSY LOUIS V,2024,149190.0,84683.0,12550.0


---
## Summary — What just happened

| Step | What we did | Why |
|------|-------------|-----|
| Found latest folder | `sorted(iterdir())[-1]` | No manual path editing — reproducible |
| Read chunks | `pd.read_parquet()` per file | Fast, type-safe |
| Concatenated | `pd.concat()` | One DataFrame for the whole panel |
| Dropped duplicates | `drop_duplicates()` | Avoid inflated row counts |
| Dropped missing IDs | `dropna(subset=[gvkey, fyear])` | Panel must be identifiable |
| Converted dtypes | `pd.to_numeric(..., errors='coerce')` | Numbers stored as numbers |
| Sorted | `sort_values([gvkey, fyear])` | Standard panel structure |
| Saved | `to_parquet()` | Compressed, fast, type-safe |

**Your full script `02_clean.py` does exactly this** — the only difference is it runs on all global firms, not just 5.

**Next session:** we use this panel in `03_descriptives.py` to build summary statistics and figures.

---
*ExInt II · WU Vienna · SS 2026 · github.com/vkiefner/sme-intl*